In [2]:
from surprise import Dataset, Reader, KNNBasic, accuracy
import pandas as pd
from surprise.model_selection import train_test_split

In [3]:
# Read the movie ratings data as CSV
columns = ["userId", "movieId", "categoryId", "reviewId", "movieRating", "reviewDate"]
df = pd.read_csv("../data/movie-ratings.txt", header=None, names=columns)
df.head()

,userId,movieId,categoryId,reviewId,movieRating,reviewDate
0,1,1,1,1,5,2000-07-12
1,2,1,1,2,5,2000-07-12
2,3,1,1,3,5,2000-07-12
3,4,1,1,4,4,2000-07-12
4,5,1,1,5,4,2000-07-12


In [4]:
reader = Reader(rating_scale=(1, 5))

data = Dataset.load_from_df(df[['userId', 'movieId', 'movieRating']], reader)
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

In [5]:
sim_options = {
    'name': 'cosine',
    'user_based': True
}

model_user_cosine = KNNBasic(k=40, sim_options=sim_options)
model_user_cosine.fit(trainset)

Computing the cosine similarity matrix...
Done computing similarity matrix.


In [6]:
sim_options = {
    'name': 'pearson',
    'user_based': True
}

model_user_pearson = KNNBasic(k=40, sim_options=sim_options)
model_user_pearson.fit(trainset)

Computing the pearson similarity matrix...
Done computing similarity matrix.


In [7]:
predictions_user_cosine = model_user_cosine.test(testset)
predictions_user_pearson = model_user_pearson.test(testset)

print("Cosine Metrics")
rmse_user_cosine = accuracy.rmse(predictions_user_cosine)
mae_user_cosine = accuracy.mae(predictions_user_cosine)

print("\nPearson Metrics")
rmse_user_pearson = accuracy.rmse(predictions_user_pearson)
mae_user_pearson = accuracy.mae(predictions_user_pearson)

Cosine Metrics
RMSE: 1.0839
MAE:  0.8200

Pearson Metrics
RMSE: 1.1034
MAE:  0.8433


In [12]:
pd.DataFrame({
    "MAE": [mae_user_cosine, mae_user_pearson],
    "RMSE": [rmse_user_cosine, rmse_user_pearson]
}, index=["Cosine", "Pearson"]).to_csv("../results/UserBasedCF_Results.csv")